# PiShield — a Shield Layer for propositional requirements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/hierarchical-requirements/examples/shield_layer_propositional.ipynb)

This notebook compares a plain neural network (**NN**) with the same network wrapped in a
PiShield Shield Layer (**NN+PiShield**) on a small, visual 2D task, reproducing the spirit of
Figure 3 of CCN+ [2]. It shows the two things the Shield Layer does: **enforcement** and
**delegation**.

We have three classes: **A₁** and **A₂** are *subclasses* of **A**, and A is exactly their
union. This is captured by three propositional requirements — two positive and one with
**negation**:

- `A :- A₁` and `A :- A₂` — whenever A₁ or A₂ holds, A must hold too;
- `A₂ :- A, ¬A₁` — whenever A holds but A₁ does not, A₂ must hold.

- **Enforcement.** The plain **NN** breaks these rules near the decision boundaries;
  **NN+PiShield** satisfies all three — including the one with negation — for *any* input,
  by construction.
- **Delegation.** Because those classes are guaranteed to follow from the others, the network
  can *delegate* them rather than learn everything directly — A is derived as A₁ ∪ A₂, and A₂ as
  "A but not A₁" (delegation through negation).

> Requires `torch`, `matplotlib` and `tqdm` (all preinstalled on Colab).

## Setup

Uses a local PiShield checkout if you're running inside one; otherwise installs PiShield
(e.g. on a fresh Colab runtime). No datasets are downloaded — the task is generated in-notebook.

In [ ]:
import importlib.util, subprocess, sys, os

root = os.path.abspath(os.getcwd())
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'pishield')):
    root = os.path.dirname(root)
if os.path.isdir(os.path.join(root, 'pishield')):
    sys.path.insert(0, root)
    print('Using local PiShield at', root)
elif importlib.util.find_spec('pishield') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/mihaela-stoian/PiShield.git@hierarchical-requirements'],
                   check=True)
    print('Installed PiShield')
else:
    print('PiShield already available')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from tqdm.auto import tqdm

from pishield.shield_layer import build_shield_layer

torch.manual_seed(0)
np.random.seed(0)

## 1. A 2D task with a class hierarchy

Points live in the unit square. **A₂** is a big ellipse, **A₁** is a smaller one elsewhere, and
**A** is their union (`A = A₁ ∪ A₂`). Each point gets three binary labels, in the order
`[A, A₁, A₂]`. Below we plot the sampled points together with the two ellipses that generate
the labels.

In [ ]:
A1_CENTER, A1_R = (0.62, 0.80), (0.13, 0.13)   # small ellipse
A2_CENTER, A2_R = (0.30, 0.40), (0.22, 0.26)   # big ellipse

def make_labels(P):
    x, y = P[:, 0], P[:, 1]
    in_A2 = ((x - A2_CENTER[0]) / A2_R[0]) ** 2 + ((y - A2_CENTER[1]) / A2_R[1]) ** 2 < 1
    in_A1 = ((x - A1_CENTER[0]) / A1_R[0]) ** 2 + ((y - A1_CENTER[1]) / A1_R[1]) ** 2 < 1
    in_A = in_A1 | in_A2                                            # A = A1 or A2
    return np.stack([in_A, in_A1, in_A2], 1).astype(np.float32)

N = 4000
X_train = np.random.rand(N, 2).astype(np.float32)
Y_train = make_labels(X_train)
X_t, Y_t = torch.tensor(X_train), torch.tensor(Y_train)
print('label counts [A, A1, A2]:', Y_train.sum(0).astype(int))

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.6))
P, L = X_train[:1500], Y_train[:1500]   # a subset, for clarity
ax.scatter(P[L[:, 0] == 0, 0], P[L[:, 0] == 0, 1], s=6, c='#d9d9d9', label='not A')
ax.scatter(P[L[:, 2] == 1, 0], P[L[:, 2] == 1, 1], s=6, c='#3f72c4', label='A2  (⊂ A)')
ax.scatter(P[L[:, 1] == 1, 0], P[L[:, 1] == 1, 1], s=6, c='#e0812f', label='A1  (⊂ A)')
ax.add_patch(Ellipse(A2_CENTER, 2 * A2_R[0], 2 * A2_R[1], fill=False, ec='#26457a', lw=2))
ax.add_patch(Ellipse(A1_CENTER, 2 * A1_R[0], 2 * A1_R[1], fill=False, ec='#9c4d10', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
ax.set_title('Training data and the shapes that generate it')
plt.show()

## 2. The requirements and the Shield Layer

With variables `A=0`, `A₁=1`, `A₂=2`, the three requirements are written as Horn rules:

- `0 :- 1` — `A :- A₁` (A holds if A₁ holds),
- `0 :- 2` — `A :- A₂` (A holds if A₂ holds),
- `2 :- 0 n1` — `A₂ :- A, ¬A₁` (A₂ holds if A holds and A₁ does **not**; the `n` prefix negates a literal).

We order the variables **A₁, A₂, A** (`custom_ordering='1,2,0'`); the layer uses the ordering to
decide how to correct and resolves the dependency between the rules automatically.

In [ ]:
constraints_path = 'shapes_constraints.txt'
with open(constraints_path, 'w') as f:
    f.write('0 :- 1\n')      # A  holds if A1 holds
    f.write('0 :- 2\n')      # A  holds if A2 holds
    f.write('2 :- 0 n1\n')   # A2 holds if A holds and A1 does NOT (negation)

shield = build_shield_layer(3, constraints_path,
                            ordering_choice='given', custom_ordering='1,2,0',
                            requirements_type='propositional')

## 3. A small classifier

A plain MLP mapping a 2D point to three per-class probabilities. We'll train two copies of it.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 64), nn.ReLU(),
                                 nn.Linear(64, 64), nn.ReLU(),
                                 nn.Linear(64, 3))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.net(x))

## 4. Train two models

The two models share the same architecture and training loop; the **only** difference is
whether the Shield Layer is applied to the output before the loss. `use_shield=True` enforces
the requirements during training (passing the labels as `goal`), which also lets the network
delegate classes it no longer has to learn directly.

In [ ]:
def train(use_shield, label, epochs=400, lr=3e-3):
    torch.manual_seed(0)   # same initialisation for a fair comparison
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCELoss()
    progress = tqdm(range(epochs), desc=f'training {label}')
    for _ in progress:
        opt.zero_grad()
        out = model(X_t)
        if use_shield:
            out = shield(out, goal=Y_t)   # enforce the requirements in the loss
        loss = bce(out, Y_t)
        loss.backward()
        opt.step()
        progress.set_postfix(loss=f'{loss.item():.4f}')
    return model

### NN — the plain network (no requirements)

In [ ]:
nn_model = train(use_shield=False, label='NN')

### NN + PiShield — trained through the Shield Layer

In [ ]:
shielded_model = train(use_shield=True, label='NN+PiShield')

## 5. Enforcement: zero coherence violations

On a dense grid we count violations of **both** kinds of rule: the positive subclass rules
(`A₁ > 0.5` or `A₂ > 0.5` while `A ≤ 0.5`) and the **negation** rule (`A > 0.5` and `A₁ ≤ 0.5`
while `A₂ ≤ 0.5`). The plain **NN** breaks them near the boundaries, and so does NN+PiShield's
raw network output *before* the layer — but *after* the layer NN+PiShield never does.

In [ ]:
grid = np.linspace(0, 1, 140)
XX, YY = np.meshgrid(grid, grid)
G = torch.tensor(np.stack([XX.ravel(), YY.ravel()], 1).astype(np.float32))

nn_model.eval(); shielded_model.eval()
with torch.no_grad():
    out_nn = nn_model(G)               # plain NN
    out_before = shielded_model(G)     # NN+PiShield, before the layer (raw network output)
    out_after = shield(out_before)     # NN+PiShield, after the layer (enforced)

def violations(out):
    A, A1, A2 = out[:, 0] > 0.5, out[:, 1] > 0.5, out[:, 2] > 0.5
    subclass = int((A1 & ~A).sum() + (A2 & ~A).sum())   # A1 -> A, A2 -> A
    negation = int((A & ~A1 & ~A2).sum())               # A and not A1 -> A2
    return subclass, negation

print('violations on the grid  (subclass rules, negation rule):')
print('  NN                        :', violations(out_nn))
print('  NN+PiShield, before layer :', violations(out_before))
print('  NN+PiShield, after layer  :', violations(out_after))

## 6. Decision boundaries

Blue = high confidence the point belongs to the class, red = low. The **dashed outline** in
each panel is the true shape that class is generated from (both ellipses for A, since
A = A₁ ∪ A₂).

Three views: the plain **NN**; **NN+PiShield before the layer** — the raw network output, where
you can see it *delegate* (it leaves class A incomplete, relying on the layer); and
**NN+PiShield after the layer** — coherent, and matching the shapes.

In [ ]:
def draw_true_shapes(ax, j):
    shapes = {0: [(A1_CENTER, A1_R), (A2_CENTER, A2_R)],   # A = A1 ∪ A2
              1: [(A1_CENTER, A1_R)], 2: [(A2_CENTER, A2_R)]}
    for center, r in shapes[j]:
        ax.add_patch(Ellipse(center, 2 * r[0], 2 * r[1], fill=False, ec='k', lw=1.2, ls='--'))

fig, axes = plt.subplots(3, 3, figsize=(8.6, 8.8))
col_names = ['Class A', 'Class A1', 'Class A2']
row_labels = ['NN', 'NN + PiShield\n(before layer)', 'NN + PiShield\n(after layer)']
for i, out in enumerate([out_nn, out_before, out_after]):
    for j in range(3):
        ax = axes[i, j]
        cf = ax.contourf(XX, YY, out[:, j].reshape(XX.shape).numpy(),
                         levels=np.linspace(0, 1, 21), cmap='RdBu', vmin=0, vmax=1)
        draw_true_shapes(ax, j)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')
        if i == 0:
            ax.set_title(col_names[j], fontsize=11)
        if j == 0:
            ax.set_ylabel(row_labels[i], fontsize=10)
fig.colorbar(cf, ax=axes, fraction=0.025, pad=0.02, label='confidence')
plt.show()

## 7. Takeaway

- **Enforcement** — **NN+PiShield** satisfies all three requirements everywhere, with zero
  violations — including the rule with **negation** (`A₂ :- A, ¬A₁`) — guaranteed for *any* input
  and *any* network, while the plain **NN** does not.
- **Delegation** — training through the shield lets the network offload classes it does not have
  to learn directly: A is derived as A₁ ∪ A₂, and A₂ as "A but not A₁" (*delegation through
  negation*). You state the requirements once and the model focuses on the parts it needs to learn.

The same recipe works for any propositional requirements — Horn rules `head :- body` (with
negated body literals via the `n` prefix) or disjunctive clauses like `y_0 or not y_1`.